## Foundations of Computer Science Project

* Gala Miladinovic, 907154
* Giorgio Tresoldi, 895956

LLM/GenAI was used for debugging and the explanation of what `.values` does.

First, we imported the pandas library, mounted the Google Drive where we shared the csv files and inspected the dataframe.

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
students = pd.read_csv('/content/drive/MyDrive/CS project/student_dropout_risk_2026.csv')

In [ ]:
students.head()

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,4,16.3,0,3,8,0,6,37,5.7,0
1,2,18,Other,USA,1,1,High,1,1,0,...,3,8.4,1,1,7,0,6,15,5.4,1
2,3,24,Other,USA,0,0,High,0,1,1,...,3,11.3,1,0,2,0,3,4,4.3,0
3,4,25,Female,USA,0,1,Middle,1,1,0,...,2,13.2,1,1,5,0,7,10,6.7,0
4,5,15,Female,Canada,0,0,Low,1,1,0,...,4,7.4,1,0,7,0,10,24,5.8,0


1. Extract all students that study `Online`, have an `attendance_rate` smaller than 50, and have more than 5 `missed_deadlines`. Store them in a dataframe called `at_risk`.

In [ ]:
at_risk = students[(students['study_mode'] == 'Online') & (students['attendance_rate'] < 50) & (students['missed_deadlines'] > 5)]
at_risk['missed_deadlines'].head(20)

,missed_deadlines
478,8
581,11
642,9
1216,7
1316,17
3756,9
3783,11
3927,7
3954,6
4541,7


2. Then compute, for each `course_type`, how many of those students there are and what fraction of them has dropped out. Store the result in a dataframe called `at_risk_by_course`, with columns `students` and `dropout_rate`, sorted by decreasing `dropout_rate`.

In [ ]:
at_risk_by_course = students.groupby('course_type', as_index=False).count()[['course_type', 'person_id']]
at_risk_by_course['dropout_rate'] = students.groupby('course_type')['dropout'].mean().values
at_risk_by_course

,course_type,person_id,dropout_rate
0,Arts & Humanities,18024,0.321016
1,Business,24986,0.318458
2,Health Sciences,14941,0.321130
3,Law & Social Sciences,11908,0.316930
4,STEM,30141,0.321456


`.values` strips the index from a Series, leaving only the rates as an array. Assigning it to `risk['dropout_rate']` puts each rate in the same row position as its course. This works because both groupby calls sort the courses the same way, so the positions line up.

3. Add to the dataframe the following columns:

* `pass_rate`: the ratio between `units_passed` and `units_enrolled`;
* `gpa_change`: the difference between `gpa_semester_2` and `gpa_semester_1`;
* `gpa_trend`: a string that is `"improving"` if `gpa_change` is larger than 0.5, `"declining"` if `gpa_change` is smaller than -0.5, and `"stable"` otherwise.

We defined a function for the `gpa_change` column that classifies the change in GPA in 3 classes: `improving`, `declining`, and `stable`. The function reads what is inside the row, which is applied to the whole dataframe in order to avoid the use of `for` loops. This method was done throughout the project to surpass the "*avoid explicit `for` loops*" restriction.

In [ ]:
def get_gpa_trend (gpa_change):
  if gpa_change > 0.5:
        return 'improving'
  if gpa_change < -0.5:
        return 'declining'
  return 'stable'

In [ ]:
students['pass_rate'] = students['units_passed'] / students['units_enrolled']
students['gpa_change'] = students['gpa_semester_2'] - students['gpa_semester_1']
students['gpa_trend'] = students['gpa_change'].apply(get_gpa_trend)
students

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,3,8,0,6,37,5.7,0,1.000000,-0.89,declining
1,2,18,Other,USA,1,1,High,1,1,0,...,1,7,0,6,15,5.4,1,0.785714,-0.03,stable
2,3,24,Other,USA,0,0,High,0,1,1,...,0,2,0,3,4,4.3,0,0.823529,-0.70,declining
3,4,25,Female,USA,0,1,Middle,1,1,0,...,1,5,0,7,10,6.7,0,0.846154,-0.41,stable
4,5,15,Female,Canada,0,0,Low,1,1,0,...,0,7,0,10,24,5.8,0,0.875000,-1.05,declining
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,25,Female,USA,0,0,Middle,1,1,1,...,0,4,0,6,11,5.9,0,1.000000,-0.98,declining
99996,99997,24,Female,India,0,1,Middle,0,1,1,...,3,7,1,5,21,5.5,1,0.500000,0.21,stable
99997,99998,27,Female,USA,0,0,Middle,1,1,1,...,2,2,0,6,29,5.8,0,1.000000,-0.57,declining
99998,99999,21,Male,UK,0,0,High,0,1,0,...,2,4,1,10,43,6.2,0,0.714286,-0.99,declining


4. Verify that, for every student, `units_passed + units_failed` equals `units_enrolled`; if not, report the rows violating this condition.

In [ ]:
violations=students[students['units_passed'] + students['units_failed'] != students['units_enrolled']]
violations

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend


We verified that every row satisfies the condition.

5. Compute the dropout rate for each value of `gpa_trend`, and the `person_id` of the 10 students with the largest GPA drop.

We interpreted the exercise as 2 separate requests, where one is for the `dropout_rate` for each value of `gpa_trend`, and the other shows the `person_id` of 10 students with the largest GPA drop.

In [ ]:
# dropout_rate for each value of gpa_trend

grouped=students.groupby('gpa_trend', as_index=False).count()[['gpa_trend','person_id']]
grouped['dropout_rate']=students.groupby('gpa_trend')['dropout'].mean().values
grouped = grouped.rename(columns={'person_id': 'tot_students'})
grouped

,gpa_trend,tot_students,dropout_rate
0,declining,57536,0.265086
1,improving,701,0.589158
2,stable,41763,0.391231


We have observed that students with the `"improving"` gpa trend have the highest dropout rate, whereas the `"declining"` students have the lowest one. It is also important to note that there is a large difference in the number of observations, which makes the dropout rate a less accurate measure for the `"improving"` group.

In [ ]:
# first 10 students with the largest GPA drop

first_10 = students.sort_values('gpa_change', ascending=True).head(10)
first_10[['person_id','gpa_change']]

,person_id,gpa_change
23225,23226,-2.49
96224,96225,-2.40
88489,88490,-2.37
51824,51825,-2.35
58935,58936,-2.33
46714,46715,-2.32
43035,43036,-2.30
72663,72664,-2.27
76705,76706,-2.23
23427,23428,-2.23


6.Partition the students into the following groups, according to `weekly_work_hours`: `0`, `1-10`, `11-20`, `21-30`, `more than 30` (all intervals include both extremes).

Firstly we inspected the `weekly_work_hours` column and checked the type of values used. Then, we defined a function which filters the rows into 5 different intervals and applied it to the `students` dataframe, defining a new column called `wwh_group`.

In [ ]:
students['weekly_work_hours']

,weekly_work_hours
0,24
1,0
2,16
3,0
4,0
...,...
99995,18
99996,15
99997,26
99998,0


In [ ]:
students['weekly_work_hours'].dtype

dtype('int64')

In [ ]:
def get_hours_group(weekly_work_hours):
  if weekly_work_hours == 0:
    return '0'
  elif 0 < weekly_work_hours and weekly_work_hours <= 10:
    return '1-10'
  elif 10 < weekly_work_hours and weekly_work_hours <= 20:
    return '11-20'
  elif 20 < weekly_work_hours and weekly_work_hours <= 30:
    return '21-30'
  elif weekly_work_hours > 30:
    return '30+'

In [ ]:
students['wwh_group'] = students['weekly_work_hours'].apply(get_hours_group)
students['wwh_group']

,wwh_group
0,21-30
1,0
2,11-20
3,0
4,0
...,...
99995,11-20
99996,11-20
99997,21-30
99998,0


7. Build a risk score by summing:

* 2 points if the tuition is not up to date;
* 1 point if `stress_level` is at least 8;
* 1 point if `sleep_hours` is smaller than 5;
* 1 point if the student has a part-time job with more than 20 weekly work hours;
* 2 points if `attendance_rate` is smaller than 60;
* 1 point if `missed_deadlines` is larger than 3.

For each value of the risk score, compute the number of students and the dropout rate.

Following the instructions, we defined a function which takes an initial score 0 and if some of the conditions are met adds up the number of points indicated. The score stays 0 if none of the conditions are met. Similarly to exercises 2 and 5, we computed the number of students (`tot_students`) and the dropout rate for each value of the risk score.

In [ ]:
def risk_score(row):
  score = 0

  if row['tuition_up_to_date'] == 0:
        score += 2

  if row['stress_level'] > 7:
        score += 1

  if row['sleep_hours'] < 5:
        score += 1

  if row['weekly_work_hours'] > 20:
        score += 1

  if row['attendance_rate'] < 60:
        score += 2

  if row['missed_deadlines'] > 3:
        score += 1

  return score

In [ ]:
students['risk_score'] = students.apply(risk_score, axis=1)
danger = students.groupby('risk_score', as_index=False).count()[['person_id', 'risk_score']]
danger['dropout_rate']=students.groupby('risk_score')['dropout'].mean().values
danger = danger.rename(columns={'person_id' : 'tot_students'})
danger

,tot_students,risk_score,dropout_rate
0,26955,0,0.152847
1,24115,1,0.252084
2,21984,2,0.337200
3,14557,3,0.452703
4,7613,4,0.577433
5,3320,5,0.673494
6,1149,6,0.794604
7,267,7,0.827715
8,40,8,0.900000


The results show that the highest scores are associated with the highest dropout rates. Implying that bad habits result in a higher dropout rate.

8. Read the GDP per capita dataset published by Our World in Data directly from the URL

https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true

and store it in a dataframe called `gdp`. The column `ny_gdp_pcap_pp_kd` contains the GDP per capita.

Clean the dataset, removing the rows that do not correspond to a single country (e.g. `World`, `High-income countries`). For each country keep only the row with the most recent year.



As with the previous dataset, we uploaded the GDP per capita dataset to Google Drive and loaded it into a DataFrame called `gdp`. We then displayed the first few rows to inspect its structure and identify the relevant columns.

In [ ]:
gdp = pd.read_csv('/content/drive/MyDrive/CS project/gdp-per-capita-worldbank.csv')

In [ ]:
gdp.head()

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia


To identify entries that do not correspond to individual countries, we examined the entries with missing values in the `owid_region` column. Then we inspected the other columns to check if there were other missing values.

In [ ]:
gdp[gdp['owid_region'].isna()]['entity'].unique()

array(['East Asia and Pacific (WB)', 'Europe and Central Asia (WB)',
       'European Union (27)', 'High-income countries',
       'Latin America and Caribbean (WB)', 'Low-income countries',
       'Lower-middle-income countries',
       'Middle East, North Africa, Afghanistan and Pakistan (WB)',
       'North America (WB)', 'South Asia (WB)', 'Sub-Saharan Africa (WB)',
       'Upper-middle-income countries', 'World'], dtype=object)

In [ ]:
gdp[gdp['code'].isna()]['entity'].unique()

array([], dtype=object)

In [ ]:
gdp[gdp['year'].isna()]['entity'].unique()

array([], dtype=object)

In [ ]:
gdp[gdp['ny_gdp_pcap_pp_kd'].isna()]['entity'].unique()

array([], dtype=object)

We then inspected the entities with non-null values in `owid_region` to verify that they correspond to individual countries or territories rather than aggregated groups.

In [ ]:
gdp[gdp['owid_region'].notnull()]['entity'].unique()

array(['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola',
       'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba',
       'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain',
       'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin',
       'Bermuda', 'Bhutan', 'Bolivia', 'Bosnia and Herzegovina',
       'Botswana', 'Brazil', 'Brunei', 'Bulgaria', 'Burkina Faso',
       'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde',
       'Cayman Islands', 'Central African Republic', 'Chad', 'Chile',
       'China', 'Colombia', 'Comoros', 'Congo', 'Costa Rica',
       "Cote d'Ivoire", 'Croatia', 'Curacao', 'Cyprus', 'Czechia',
       'Democratic Republic of Congo', 'Denmark', 'Djibouti', 'Dominica',
       'Dominican Republic', 'East Timor', 'Ecuador', 'Egypt',
       'El Salvador', 'Equatorial Guinea', 'Estonia', 'Eswatini',
       'Ethiopia', 'Faroe Islands', 'Fiji', 'Finland', 'France', 'Gabon',
       'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece',

In [ ]:
gdp.dropna()

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia
...,...,...,...,...,...
7440,Zimbabwe,ZWE,2021,4827.1094,Africa
7441,Zimbabwe,ZWE,2022,5036.7830,Africa
7442,Zimbabwe,ZWE,2023,5218.0454,Africa
7443,Zimbabwe,ZWE,2024,5211.8975,Africa


We grouped the data by country using `groupby('entity')` and used `idxmax()` to identify the index of the row with the most recent year for each country. We then used `.loc[]` to select these rows, keeping only the most recent observation for each country. Finally, we used `.reset_index()` to reset the DataFrame's index.

In [ ]:
most_recent_year = gdp.loc[gdp.groupby('entity')['year'].idxmax()].reset_index()
most_recent_year

,index,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,24,Afghanistan,AFG,2024,1964.4523,Asia
1,60,Albania,ALB,2025,22724.2930,Europe
2,96,Algeria,DZA,2025,15882.7140,Africa
3,132,Andorra,AND,2025,67993.0550,Europe
4,168,Angola,AGO,2025,8760.1900,Africa
...,...,...,...,...,...,...
207,7300,Vanuatu,VUT,2025,3866.9980,Oceania
208,7336,Vietnam,VNM,2025,15457.3500,Asia
209,7372,World,OWID_WRL,2025,21913.5100,NaN
210,7408,Zambia,ZMB,2025,3742.2563,Africa


9. Join the GPD data with the students data. Compute the dropout rate for each continent.

In [ ]:
gdp_students = pd.merge(gdp, students, left_on='entity', right_on='nationality')
gdp_students

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region,person_id,age,gender,nationality,first_generation,...,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend,wwh_group,risk_score
0,Australia,AUS,1990,36267.742,Oceania,181,26,Male,Australia,1,...,0,10,1,5.6,0,0.928571,-0.79,declining,0,3
1,Australia,AUS,1990,36267.742,Oceania,225,22,Female,Australia,1,...,0,6,87,7.9,0,1.000000,-1.19,declining,0,3
2,Australia,AUS,1990,36267.742,Oceania,286,20,Female,Australia,0,...,0,6,17,8.9,0,0.714286,-0.20,stable,0,0
3,Australia,AUS,1990,36267.742,Oceania,501,26,Female,Australia,0,...,1,10,14,8.5,0,1.000000,-0.89,declining,0,1
4,Australia,AUS,1990,36267.742,Oceania,526,20,Female,Australia,0,...,0,5,34,5.5,0,0.600000,-0.90,declining,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2302951,South Korea,KOR,2025,55697.086,Asia,99880,26,Other,South Korea,0,...,0,8,22,4.0,0,0.857143,-0.47,stable,11-20,2
2302952,South Korea,KOR,2025,55697.086,Asia,99881,19,Female,South Korea,0,...,0,9,0,4.0,0,0.875000,-0.77,declining,0,2
2302953,South Korea,KOR,2025,55697.086,Asia,99909,19,Female,South Korea,0,...,0,2,29,9.4,0,0.800000,-0.17,stable,11-20,1
2302954,South Korea,KOR,2025,55697.086,Asia,99949,21,Female,South Korea,0,...,0,7,103,8.6,0,1.000000,-0.88,declining,0,0


The same logic from previous exercises was used for computing the dropout rate.

In [ ]:
continental_danger = gdp_students.groupby('owid_region', as_index=False).count()[['owid_region', 'person_id']]
continental_danger['dropout_rate']=gdp_students.groupby('owid_region')['dropout'].mean().values
continental_danger = continental_danger.rename(columns={'person_id' : 'tot_students'})
continental_danger

,owid_region,tot_students,dropout_rate
0,Africa,284616,0.316721
1,Asia,1297260,0.322908
2,Europe,146592,0.325639
3,North America,292176,0.310621
4,Oceania,34884,0.321981
5,South America,247428,0.317620
